# Chunked Triton SSM Scan (check + benchmark) — T4 GPU

Validates the **chunked two-level scan** (`ChunkedSSMScanFn` in `VECTOR/triton_scan.py`) against the fused scan and a pure-PyTorch reference, then benchmarks it against the fused and old (eager exp/mul) paths.

**What to look for:**
1. `CHUNKED CHECK: PASS` — the chunked forward matches the reference and chunked grads match the fused path to `atol=1e-4`.
2. `chunked vs fused` speedup in `bench_chunked` — the two-level scan should win at large T (serial depth drops from O(T) to O(chunk + T/chunk)).

Run cells in order. Requires a **T4 GPU** runtime (Runtime → Change runtime type).

> **Failed the check?** Copy the `chunk=...` lines (fwd-vs-ref / worst grad diff) from cell 3 and paste them into the chat with the code changes.

In [ ]:
# @title 1. Environment check
import sys, os, time
import numpy as np
import torch

print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Python {sys.version.split()[0]}')

if torch.version.cuda is None:
    print('ERROR: No CUDA build of PyTorch. Use Runtime > Change runtime type > T4 GPU.')
    raise SystemExit(1)
if torch.cuda.device_count() == 0:
    print('ERROR: No GPU detected.')
    raise SystemExit(1)
print('Environment OK')

In [ ]:
# @title 2. Clone repo + load triton_scan (self-contained)
import sys, os

REPO_URL = 'https://github.com/nishantXnova/RETRANS-X.git'
PROJECT_DIR = '/content/RETRANS-X'

if not os.path.isdir(PROJECT_DIR):
    print('Cloning repo...')
    !git clone --quiet {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print('Force-updating to origin/main...')
!git fetch --quiet origin
!git reset --hard --quiet origin/main

vec_dir = os.path.join(PROJECT_DIR, 'VECTOR')
triton_path = os.path.join(vec_dir, 'triton_scan.py')
print(f'triton_scan.py exists: {os.path.isfile(triton_path)}')
if not os.path.isfile(triton_path):
    print('STILL not found.', os.listdir(vec_dir))
    raise SystemExit(1)

# Always load the freshly pulled file; register it so later cells can
# `from triton_scan import ...` (adds vec_dir to sys.path too).
sys.path.insert(0, vec_dir)
sys.modules.pop('triton_scan', None)
import triton_scan as mod

HAS_TRITON = mod.HAS_TRITON
print(f'HAS_TRITON: {HAS_TRITON}')
if not HAS_TRITON:
    print('ERROR: Triton not available on this runtime. Expected on Colab T4 with Linux.')
    raise SystemExit(1)

In [ ]:
# @title 3. Chunked scan: correctness check
# check_fused() first (the fused path is the gradient ground truth for chunked),
# then check_chunked() compares chunked fwd vs pure-PyTorch and chunked grads vs fused.
from triton_scan import check_fused, check_chunked

ok_fused = check_fused()
print()
ok_chunked = check_chunked()
print()
print('FINAL:', 'PASS' if (ok_fused and ok_chunked) else 'FAIL')

In [ ]:
# @title 4. Chunked vs fused vs old: benchmark (fwd+bwd)
from triton_scan import bench_chunked, CHUNK_DEFAULT

print(f'CHUNK_DEFAULT = {CHUNK_DEFAULT}\n')
bench_chunked(T=4096, H=128, N=8, B=4, iters=20)
print()
bench_chunked(T=16384, H=128, N=8, B=4, iters=10)

In [ ]:
# @title 5. Chunk-size sweep (optional): which C_CHUNK is fastest at T=8192?
import torch, time
from triton_scan import ChunkedSSMScanFn

device = 'cuda'
B, T, H, N = 4, 8192, 128, 8
torch.manual_seed(0)
u = torch.randn(B, T, H, device=device)
dt = (torch.rand(B, T, H, device=device).clamp_min(1e-3) * 0.1).requires_grad_()
A = (-torch.rand(H, N, device=device).clamp_min(1e-3)).requires_grad_()
Bp = torch.randn(B, T, N, device=device).requires_grad_()
C = torch.randn(B, T, N, device=device)
D = torch.randn(H, device=device)

def run(chunk):
    h = ChunkedSSMScanFn.apply(u, dt, A, Bp, T, chunk)
    y = (h * C.unsqueeze(2)).sum(-1) + D * u
    y.pow(2).mean().backward()
    for t in (dt, A, Bp): t.grad = None

print('T=%d H=%d N=%d B=%d: fwd+bwd per C_CHUNK' % (T, H, N, B))
for chunk in (32, 64, 128, 256, 512, 1024):
    if T % chunk:
        continue
    for _ in range(3):
        run(chunk)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        run(chunk)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / 10 * 1000
    print(f'  C_CHUNK={chunk:>4}: {ms:7.2f} ms')

In [ ]:
# @title 6. THE regime test: B=1, long T
# If chain depth is the bottleneck, chunked should win HERE (fused collapses
# to B*H=128 programs at B=1/T=65536). B=4/T<=16k was the wrong regime to bench it.
import torch, time
from triton_scan import bench_chunked, ChunkedSSMScanFn

print('=== B=1, T=32768 ===')
bench_chunked(T=32768, H=128, N=8, B=1, iters=10)
print()
print('=== B=1, T=65536 ===')
bench_chunked(T=65536, H=128, N=8, B=1, iters=5)
print()

device = 'cuda'
B, T, H, N = 1, 65536, 128, 8
torch.manual_seed(0)
u = torch.randn(B, T, H, device=device)
dt = (torch.rand(B, T, H, device=device).clamp_min(1e-3) * 0.1).requires_grad_()
A = (-torch.rand(H, N, device=device).clamp_min(1e-3)).requires_grad_()
Bp = torch.randn(B, T, N, device=device).requires_grad_()
C = torch.randn(B, T, N, device=device)
D = torch.randn(H, device=device)

def run(chunk):
    h = ChunkedSSMScanFn.apply(u, dt, A, Bp, T, chunk)
    y = (h * C.unsqueeze(2)).sum(-1) + D * u
    y.pow(2).mean().backward()
    for t in (dt, A, Bp): t.grad = None

print('T=%d H=%d N=%d B=%d: fwd+bwd per C_CHUNK' % (T, H, N, B))
for chunk in (64, 128, 256, 512, 1024, 2048):
    if T % chunk:
        continue
    for _ in range(2):
        run(chunk)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(5):
        run(chunk)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / 5 * 1000
    print(f'  C_CHUNK={chunk:>5}: {ms:7.2f} ms')

In [ ]:
# @title 7. Final pin: B*H=192 (last unmeasured crossover point)
# RESOLVED: B*H is the driver (B=2,H=64 wins 1.18x), NOT B; no T floor in
# the winning regime (B=1 wins at T=4096/8192 too). Thresholds now
# AUTO_MAX_BH=128, AUTO_MIN_T dropped entirely. Crossover sits in (128, 256): pin 192.
#   -> chunked wins here -> raise AUTO_MAX_BH to 192
#   -> fused wins here   -> keep 128
from triton_scan import bench_chunked

print('=== Final pin: B*H=192 — raise AUTO_MAX_BH to 192? ===')
print('  B=1, H=192, T=32768:')
bench_chunked(T=32768, H=192, N=8, B=1, iters=10)
print()
print('  B=3, H=64, T=32768 (also B*H=192):')
bench_chunked(T=32768, H=64, N=8, B=3, iters=10)